# Process Multiple Long Strips

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import papermill

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [3]:
#%% Test Record Excel
fname_excel = r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx';
dftests = pd.read_excel(
    fname_excel,
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Unnamed: 24
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,64.0,2025-05-12,GHL_pyapp_20250512T1258,0.350-0.374mg/mm lrg bag blu,ES0331#33 0.373,0.373,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runD,Simon
70,65.0,2025-05-12,GHL_pyapp_20250512T1303,0.350-0.374mg/mm lrg bag blu,ES0414#1 0.353,0.353,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runD,Simon
71,66.0,2025-05-12,GHL_pyapp_20250512T1341,Valves ES 4/16/2024 1-sided no reflo,#1,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,trimmed ends to fit
72,67.0,2025-05-12,GHL_pyapp_20250512T1350,Valves ES 4/16/2024 1-sided no reflo,#2,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,trimmed ends to fit


In [6]:
#%% Filter The Tests To Process
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     dftests['Test date']=='2025-05-05',
# ]
# dfmasks = [
#     dftests['Test date']=='2025-05-08',
# ]
dfmasks = [
    dftests['Test date']=='2025-05-12',
    ~dftests['Batch'].str.contains('Valves ES')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Unnamed: 24
38,33.0,2025-05-12,GHL_pyapp_20250512T0947,0.350-0.374mg/mm lrg bag blu,ES0414#7 0.367,0.367,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rem run,Simon
39,34.0,2025-05-12,GHL_pyapp_20250512T0951,0.350-0.374mg/mm lrg bag blu,ES0414#5 0.352,0.352,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rem run,Simon
40,35.0,2025-05-12,GHL_pyapp_20250512T0955,0.350-0.374mg/mm lrg bag blu,ES0415#8 0.351,0.351,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rem run,Simon
41,36.0,2025-05-12,GHL_pyapp_20250512T0959,0.350-0.374mg/mm lrg bag blu,ES0408#16 0.372,0.372,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rem run,Simon
42,37.0,2025-05-12,GHL_pyapp_20250512T1002,0.350-0.374mg/mm lrg bag blu,ES0409#7 0.366,0.366,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rem run,Simon
43,38.0,2025-05-12,GHL_pyapp_20250512T1117,0.350-0.374mg/mm lrg bag blu,ES0410#2 0.372,0.372,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runC,Simon
44,39.0,2025-05-12,GHL_pyapp_20250512T1121,0.350-0.374mg/mm lrg bag blu,ES0415#30 0.359,0.359,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runC,Simon
45,40.0,2025-05-12,GHL_pyapp_20250512T1125,0.350-0.374mg/mm lrg bag blu,ES0415#27 0.354,0.354,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runC,Simon
46,41.0,2025-05-12,GHL_pyapp_20250512T1130,0.350-0.374mg/mm lrg bag blu,ES0415#26 0.374,0.374,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runC,Simon
47,42.0,2025-05-12,GHL_pyapp_20250512T1133,0.350-0.374mg/mm lrg bag blu,ES0415#22 0.366,0.366,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,loc runC,Simon


In [7]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


STUDY: GHL_pyapp_20250512T0947
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250512T0951
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250512T0955
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250512T0959
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250512T1002
{   'study_has_an_ini_file': True,
    'study_has_json_info_f

# Manipulate excel directly

In [8]:
import openpyxl

workbook = openpyxl.load_workbook(fname_excel);

In [9]:
sheet = workbook.active

In [10]:
hdrrow = 2;
hdrcols = sheet[hdrrow];
print(hdrcols)
hdr = {c.value:cnt+1 for cnt,c in enumerate(hdrcols)};
print(hdr)

(<Cell 'Strip measurement'.A2>, <Cell 'Strip measurement'.B2>, <Cell 'Strip measurement'.C2>, <Cell 'Strip measurement'.D2>, <Cell 'Strip measurement'.E2>, <Cell 'Strip measurement'.F2>, <Cell 'Strip measurement'.G2>, <Cell 'Strip measurement'.H2>, <Cell 'Strip measurement'.I2>, <Cell 'Strip measurement'.J2>, <Cell 'Strip measurement'.K2>, <Cell 'Strip measurement'.L2>, <Cell 'Strip measurement'.M2>, <Cell 'Strip measurement'.N2>, <Cell 'Strip measurement'.O2>, <Cell 'Strip measurement'.P2>, <Cell 'Strip measurement'.Q2>, <Cell 'Strip measurement'.R2>, <Cell 'Strip measurement'.S2>, <Cell 'Strip measurement'.T2>, <Cell 'Strip measurement'.U2>, <Cell 'Strip measurement'.V2>, <Cell 'Strip measurement'.W2>, <Cell 'Strip measurement'.X2>, <Cell 'Strip measurement'.Y2>)
{None: 25, 'Test ID': 2, 'Test date': 3, 'Test name': 4, 'Batch': 5, 'Strip': 6, 'Wax (mg/mm)': 7, 'Fixture': 8, 'X, FOV': 9, 'Y, FOV': 10, 'Z, FOV': 11, 'X, pixel size': 12, 'Y, pixel size': 13, 'Z, pixel size': 14, 'Angle'

In [15]:
for row in sheet.iter_rows(min_row=hdrrow+3,min_col=hdrcols[0].column):
    record = {hdrcols[cnt].value:c.value for cnt,c in enumerate(row)}
    matchingoctstudy = [octstudy for octstudy in octstudies if octstudy.name==record["Test name"]]
    if len(matchingoctstudy)==1:
        print(record)
        print(octstudy.name)
        octstudy = matchingoctstudy[0]
        octstudy.load_first_oct();
        octdata = octstudy.octdatalist[0]
        cfg_oct_xml = octdata.cfg_oct_xml;

        # populate filesize
        row[hdr['Total size (GB)']-1].value = round(sum([f.stat().st_size for f in octstudy.folder_study.glob('*.oct')])/1e9,2);
        row[hdr['File size (MB)']-1].value = round((sum([f.stat().st_size for f in octstudy.folder_study.glob('*.oct')])/1e9)/octstudy.num_oct_files,2);

        # populate acquisition details
        pixelsize_x = float(cfg_oct_xml.Ocity.Image.PixelSpacing.SizeX.etElem.text)
        pixelsize_y = float(cfg_oct_xml.Ocity.Image.PixelSpacing.SizeY.etElem.text)
        pixelsize_z = float(cfg_oct_xml.Ocity.Image.PixelSpacing.SizeZ.etElem.text)
        totalsizemm_x = float(cfg_oct_xml.Ocity.Image.SizeReal.SizeX.etElem.text)
        totalsizemm_y = float(cfg_oct_xml.Ocity.Image.SizeReal.SizeY.etElem.text)
        totalsizemm_z = float(cfg_oct_xml.Ocity.Image.SizeReal.SizeZ.etElem.text)
        row[hdr['X, FOV']-1].value = totalsizemm_x;
        row[hdr['Y, FOV']-1].value = totalsizemm_y;
        row[hdr['Z, FOV']-1].value = totalsizemm_z;
        row[hdr['X, pixel size']-1].value = pixelsize_x*1e3;
        row[hdr['Y, pixel size']-1].value = pixelsize_y*1e3;
        row[hdr['Z, pixel size']-1].value = pixelsize_z*1e3;
        row[hdr['Angle']-1].value = float(cfg_oct_xml.Ocity.Image.Angle.etElem.text)
        row[hdr['Speed/Sensitivity']-1].value = cfg_oct_xml.Ocity.Instrument.DevicePresetDescription.etElem.text;
        row[hdr['Averaging (A-scan)']-1].value = int(cfg_oct_xml.Ocity.Acquisition.IntensityAveraging.AScans.etElem.text)
        row[hdr['Refractive Index']-1].value = round(float(cfg_oct_xml.Ocity.Acquisition.RefractiveIndex.etElem.text),1);

        if octstudy.has_json_info_file:
            jsoninfo = octstudy.json_info_file;
            row[hdr['Left edge (mm)']-1].value = jsoninfo['pos_leftmost_edge_center'];
            row[hdr['Right edge (mm)']-1].value = jsoninfo['pos_rightmost_edge_center'];
            row[hdr['Strip length (mm)']-1].value = jsoninfo['pos_rightmost_edge_center']-jsoninfo['pos_leftmost_edge_center'];

        rescheck = octstudy.resultsCheck();
        haveAllResults = not all([v is False for k,v in rescheck.items()])
        #haveAllResults = haveAllResults and (len(rescheck['along_strip_data_extracted'])==5)
        print('results done?',haveAllResults)
        if(haveAllResults):
            row[hdr['ProcessingNotes']-1].value = 'done,{:d}'.format(len(rescheck['along_strip_data_extracted']))
        else:
            row[hdr['ProcessingNotes']-1].value = ''
        #break

    #octstudy.octdatalist[0]

{None: 'Simon', 'Test ID': 33, 'Test date': datetime.datetime(2025, 5, 12, 0, 0), 'Test name': 'GHL_pyapp_20250512T0947', 'Batch': '0.350-0.374mg/mm lrg bag blu', 'Strip': 'ES0414#7 0.367', 'Wax (mg/mm)': '=_xlfn.NUMBERVALUE(_xlfn.TEXTAFTER(F41, " "))', 'Fixture': 'Vacuum', 'X, FOV': 9.0, 'Y, FOV': 3.5, 'Z, FOV': 2.39775, 'X, pixel size': 20.0, 'Y, pixel size': 20.0, 'Z, pixel size': 3.4749999999999996, 'Angle': 0.0, 'Right edge (mm)': 308.5, 'Left edge (mm)': 143.0, 'Strip length (mm)': 165.5, 'Speed/Sensitivity': 'Default (Medium sensitivity, 48 kHz)', 'Averaging (A-scan)': 2, 'Refractive Index': None, 'File size (MB)': 0.21816383999999997, 'Total size (GB)': 4.14511296, 'ProcessingNotes': 'done,5'}
GHL_pyapp_20250512T1303
Image Dims:(690, 450, 175) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
results done? True
{None: 'Simon', 'Test ID': 34, 'Test date': datetime.datetime(2025, 5, 12, 0, 0), 'Test name': 'GHL_pyapp_20250512T0951', 'Batch': '0.350-0.374mg/mm lrg bag blu', 'Strip': 'ES0414

In [14]:
round(float(cfg_oct_xml.Ocity.Acquisition.RefractiveIndex.etElem.text),1)

1.6

In [18]:
hdr

{None: 24,
 'Test ID': 2,
 'Test date': 3,
 'Test name': 4,
 'Batch': 5,
 'Strip': 6,
 'Wax (mg/mm)': 7,
 'Fixture': 8,
 'X, FOV': 9,
 'Y, FOV': 10,
 'Z, FOV': 11,
 'X, pixel size': 12,
 'Y, pixel size': 13,
 'Z, pixel size': 14,
 'Angle': 15,
 'Right edge (mm)': 16,
 'Left edge (mm)': 17,
 'Strip length (mm)': 18,
 'Speed/Sensitivity': 19,
 'Averaging (A-scan)': 20,
 'File size (MB)': 21,
 'Total size (GB)': 22,
 'ProcessingNotes': 23}

In [28]:
len(octstudy.resultsCheck()['along_strip_data_extracted'])==5

True

In [48]:
from xml.etree import ElementTree
ElementTree.dump(cfg_oct_xml.getroot())

<Ocity version="1.0">
    <DataFiles>
        <DataFile Type="Colored" SizeZ="648" SizeX="484" RangeZ="648" RangeX="484" RangeY="1" BytesPerPixel="4">data\VideoImage.data</DataFile>
        <DataFile Type="Colored" SizeZ="640" SizeX="480" RangeZ="1" RangeX="1" RangeY="1" BytesPerPixel="4">data\PreviewImage.data</DataFile>
        <DataFile Type="Real" SizeZ="690" SizeX="450" SizeY="175" RangeZ="1.4985937499999997" RangeX="9" RangeY="3.5" BytesPerPixel="4">data\Intensity.data</DataFile>
        <DataFile Type="Text">data\Probe.ini</DataFile>
    </DataFiles>
    <Image Type="Processed">
        <SizePixel Unit="px">
            <SizeZ>690</SizeZ>
            <SizeX>450</SizeX>
            <SizeY>175</SizeY>
        </SizePixel>
        <SizeReal Unit="mm">
            <SizeZ>2.397750</SizeZ>
            <SizeX>9.000000</SizeX>
            <SizeY>3.500000</SizeY>
        </SizeReal>
        <PixelSpacing>
            <SizeZ>0.003475</SizeZ>
            <SizeX>0.020000</SizeX>
           

In [16]:
workbook.save(fname_excel)

In [62]:
cfg_oct_xml.Ocity.Instrument.DevicePresetDescription.etElem.text

'Default (Medium sensitivity, 48 kHz)'

In [17]:
octstudy.json_info_file['pos_leftmost_edge_center']

143.3